# Feature notebook

This notebook reads Postgres credentials from the project's `.env` file .


In [1]:
import pandas as pd
from notebook_common import DB_CONFIG, ENV_PATH, get_connection_check_df, query_postgres

In [2]:
print(f"Loaded credentials from {ENV_PATH}")
connection_check_df = get_connection_check_df()
connection_check_df

Loaded credentials from /home/ruthikreddy/Desktop/may_29/.env


,database_name,schema_name,database_user,connected_at
0,tulip2,public,postgres,2026-05-29 16:25:50.376659+05:30


## Read from `dak`

In [ ]:
dak_df = query_postgres("""
SELECT *
FROM dak
WHERE list_date < '2026-01-01'
""")

In [4]:
dak_df.shape

(1407551, 73)

In [5]:
dak_df.isnull().sum()

id                               0
list_no                       2015
dakid_no                       170
fk_section                       0
fk_dak_type                      0
                            ...   
uuid                       1407551
fk_central_unit              89905
fk_central_vendor          1381150
fk_central_civ_employee    1358903
file_path                  1407551
Length: 73, dtype: int64

In [7]:
def filter_columns(
    df: pd.DataFrame,
    schema_df: pd.DataFrame,
    *,
    null_threshold: float = 80,
    varchar_length_threshold: int = 10,
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    null_percentage = df.isnull().mean().mul(100)
    high_null_columns = null_percentage[null_percentage >= null_threshold].index.tolist()
    working_df = df.drop(columns=high_null_columns)

    '''id_like_columns = [
        column
        for column in working_df.columns
        if column == "id" or column.endswith("_id") or column.startswith("id_")
    ]'''

    text_like_columns = schema_df.loc[
        schema_df["data_type"].isin(["text"]),
        "column_name",
    ].tolist()
    text_like_columns = [column for column in text_like_columns if column in working_df.columns]

    long_varchar_columns = schema_df.loc[
        (schema_df["data_type"] == "character varying")
        & (schema_df["character_maximum_length"].fillna(0) > varchar_length_threshold),
        "column_name",
    ].tolist()
    long_varchar_columns = [column for column in long_varchar_columns if column in working_df.columns]

    columns_to_drop = sorted(set(text_like_columns + long_varchar_columns))
    filtered_df = working_df.drop(columns=columns_to_drop, errors="ignore")

    dropped_summary = {
        "high_null_columns": sorted(set(high_null_columns)),
        #"id_like_columns": sorted(set(id_like_columns)),
        "text_like_columns": sorted(set(text_like_columns)),
        "long_varchar_columns": sorted(set(long_varchar_columns)),
        "final_dropped_columns": sorted(set(high_null_columns + columns_to_drop)),
    }
    return filtered_df, dropped_summary


In [8]:
def get_table_schema(table_name: str) -> pd.DataFrame:
    return query_postgres(f"""
    SELECT
        column_name,
        data_type,
        udt_name,
        character_maximum_length
    FROM information_schema.columns
    WHERE table_schema = '{DB_CONFIG['schema']}'
      AND table_name = '{table_name}'
    ORDER BY ordinal_position
    """)


In [9]:
dak_schema_df = get_table_schema("dak")
dak_schema_df.head()

,column_name,data_type,udt_name,character_maximum_length
0,id,bigint,int8,NaN
1,list_no,integer,int4,NaN
2,dakid_no,character varying,varchar,18.0
3,fk_section,integer,int4,NaN
4,fk_dak_type,integer,int4,NaN


In [10]:
filtered_df, dropped_columns_summary = filter_columns(
    dak_df,
    dak_schema_df,
    varchar_length_threshold=10,
)

dak_filtered_df = filtered_df

print(f"Original shape: {dak_df.shape}")
print(f"Filtered shape: {dak_filtered_df.shape}")


Original shape: (1407551, 73)
Filtered shape: (1407551, 23)


In [11]:
pd.Series({key: len(value) for key, value in dropped_columns_summary.items()})

high_null_columns        42
text_like_columns         2
long_varchar_columns      6
final_dropped_columns    50
dtype: int64

In [12]:
dropped_columns_summary

{'high_null_columns': ['aao_disposal_date',
  'ao_disposal_date',
  'auditor_disposal_date',
  'codehead',
  'dad_account_no',
  'emp_no',
  'exp_cat',
  'file_path',
  'fis_code_head',
  'fis_date',
  'fis_doc_no',
  'fis_imported_at',
  'fis_xml_file_no',
  'fk_aao_disposal',
  'fk_ao_disposal',
  'fk_auditor_disposal',
  'fk_bill_type',
  'fk_central_civ_employee',
  'fk_central_vendor',
  'fk_civ_employee',
  'fk_dad_employee',
  'fk_dad_office',
  'fk_fis_imported_by',
  'fk_imprest',
  'fk_old_dak',
  'fk_outward_dak',
  'fk_usr',
  'fk_vendor',
  'gpf_pran_ppan_no',
  'major_head',
  'make_cat',
  'msme_cat',
  'pfms_sanction_date',
  'pfms_sanction_id',
  'priority',
  'regn_no',
  'remarks',
  'sender_city',
  'sender_name',
  'task_no',
  'temp_dak_id',
  'uuid'],
 'text_like_columns': ['reason', 'subject'],
 'long_varchar_columns': ['bill_no',
  'dakid_no',
  'disposal_no',
  'mode_of_receipt',
  'reference_no',
  'security_grading'],
 'final_dropped_columns': ['aao_disposal

In [13]:
dak_filtered_df.head()

,id,list_no,fk_section,fk_dak_type,fk_unit,reference_date,bill_date,amount,disposal_date,record_status,...,created_at,fk_task_usr,fk_office_id,month_year,dak_year,list_date,gem_bill,pfms_bill,kpi_bill_type,fk_central_unit
0,83689,0.0,25,528,1214.0,2022-06-21,2022-06-21,1366513.0,2022-06-27,D,...,2020-09-30 06:30:39.081,1427.0,53,06/2022,2022-2023,2020-09-30,NaN,NaN,NaN,1214.0
1,83676,0.0,25,528,1214.0,2022-06-21,2022-06-21,5405258.0,2022-06-27,D,...,2020-09-30 06:28:57.306,1427.0,53,06/2022,2022-2023,2020-09-30,NaN,NaN,NaN,1214.0
2,83652,0.0,25,528,1214.0,2022-06-20,2022-06-20,25655564.0,2022-06-27,D,...,2020-09-30 06:26:08.55,1427.0,53,06/2022,2022-2023,2020-09-30,NaN,NaN,NaN,1214.0
3,83645,0.0,25,528,1214.0,2022-06-20,2022-06-20,1089980.0,2022-06-27,D,...,2020-09-30 06:25:38.714,1427.0,53,06/2022,2022-2023,2020-09-30,NaN,NaN,NaN,1214.0
4,83668,0.0,25,528,1214.0,2022-06-21,2022-06-21,1227701.0,2022-06-27,D,...,2020-09-30 06:28:08.802,1427.0,53,06/2022,2022-2023,2020-09-30,NaN,NaN,NaN,1214.0


In [14]:
dak_filtered_df.isnull().sum()

id                       0
list_no               2015
fk_section               0
fk_dak_type              0
fk_unit              89905
reference_date         162
bill_date           510547
amount              318215
disposal_date       162962
record_status            0
fk_dak_entry_usr    270442
rescheduled         270968
multiple_entry           0
created_at               0
fk_task_usr         270686
fk_office_id             0
month_year               0
dak_year                 0
list_date                0
gem_bill            426674
pfms_bill           813325
kpi_bill_type       632979
fk_central_unit      89905
dtype: int64

In [19]:
dak_filtered_df.to_csv("dak_filtered.csv", index=False)